# Setup

In [ ]:
# load required libraries with requirements.txt
! pip install -r requirements.txt

In [3]:
#Initial imports

import matplotlib.pyplot as plt
import mysql.connector as mysql
import pandas as pd
import numpy as np
import json

from DbConnector import DbConnector

In [ ]:
# Import datasets
credits_df = pd.read_csv("movies_ds/credits.csv")
keywords_df = pd.read_csv("movies_ds/keywords.csv")
links_small_df = pd.read_csv("movies_ds/links_small.csv")
links_df = pd.read_csv("movies_ds/links.csv")
movies_metadata_df = pd.read_csv("movies_ds/movies_metadata.csv")
ratings_small_df = pd.read_csv("movies_ds/ratings_small.csv")
ratings_df = pd.read_csv("movies_ds/ratings.csv")

In [ ]:
# Cleaning of the data
credits_df_clean = credits_df.copy()
keywords_df_clean = keywords_df.copy()
links_small_df_clean = links_small_df.copy()
links_df_clean = links_df.copy()
movies_metadata_df_clean = movies_metadata_df.copy()
ratings_small_df_clean = ratings_small_df.copy()
ratings_df_clean = ratings_df.copy()

# Getting rid of duplicates (movies with the same id)
movies_metadata_df_clean = movies_metadata_df_clean.drop_duplicates(subset='id')

# Getting rid of movies with unknown release date
movies_metadata_df_clean = movies_metadata_df_clean.dropna(axis=0, subset='release_date')

In [ ]:
from datetime import datetime

def timestamp_to_datetime(timestamp):
    """
    In: timestamp (number of seconds since January 1st, 1970)
        typically items from the 'timestamp' column in the ratings_df dataset are in this format
    Out: datetime following this format: YYYY-MM-DD HH:MM:SS
    """
    return datetime.fromtimestamp(timestamp)

def date_to_datetime(date):
    """
    In: date following this format: YYYY-MM-DD as a string
        typically items from the 'release_date' column in the movies_metadata_df dataset are in this format
    Out: datetime following this format: YYYY-MM-DD HH:MM:SS
    """
    return datetime.strptime(date, "%Y-%m-%d")

# Getting rid of reviews posted before the movie release date
# Only doable for ratings_small_df, takes too long to compute for ratings_df
nb_ratings_deleted = 0
print(len(ratings_small_df_clean))
i = 0
for index, row in ratings_small_df_clean.iterrows():
    rating_datetime = timestamp_to_datetime(row['timestamp'])
    movie_id = str(int(row['movieId']))
    movie = movies_metadata_df_clean.loc[movies_metadata_df_clean['id'] == movie_id]
    if not movie.empty:
        movie = movie.iloc[0]
        movie_release_datetime = date_to_datetime(str(movie['release_date']))
        if rating_datetime < movie_release_datetime:
            ratings_small_df_clean.drop(index)
            nb_ratings_deleted += 1
    i += 1
    print(i)
print(str(nb_ratings_deleted) + " ratings deleted")